# 09 — Rule Agent (`agents/rule_agent.py`)
Manages DQ (Data Quality) and Business rules in an in-memory registry.  
Three operations: **list**, **create**, **evaluate**

Pre-loaded rules:
- `DQ-001` — Retention Completeness Check  
- `DQ-002` — Retention Rate Range Validity  
- `BR-001` — GRR Minimum Threshold Alert  
- `BR-002` — CAC Payback Period Ceiling


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'codebase', 'codebase', 'src'))
os.makedirs("logs", exist_ok=True)

## 1. Inspect the Pre-Loaded Rule Registry

In [ ]:
from agents.rule_agent import RULE_REGISTRY
import json
print(f"Registry has {len(RULE_REGISTRY)} rules:\n")
for rule_id, rule in RULE_REGISTRY.items():
    print(f"  [{rule['type']}] {rule_id}: {rule['name']}")
    print(f"    asset={rule['asset']} | expr={rule['expression']} | severity={rule['severity']}")


## 2. List All Rules

In [ ]:
from agents.rule_agent import RuleAgent
from core.base_agent import AgentRequest

agent = RuleAgent()
request = AgentRequest(query="list rules")
result = agent.execute(request)

print(result.summary)
print("\n--- Result Metadata ---")
print("total_rules:", result.metadata["total_rules"])

## 3. List Rules Filtered by Data Product

In [ ]:
# Only retention-related rules
req = AgentRequest(query="show rules for retention", data_products=["retention"])
result = agent.execute(req)
print(result.summary)

## 4. Create a Data Quality Rule

In [ ]:
req = AgentRequest(
    query="create dq rule for null check on bookings",
    context={
        "rule_name": "Bookings Null Check",
        "dimension": "completeness",
        "asset": "analytics.bookings_fact",
        "expression": "null_count / total_count < 0.02",
        "threshold": 0.02,
        "severity": "High",
        "owner": "Revenue Operations",
    },
    data_products=["bookings"],
)
result = agent.execute(req)
print(result.summary)
print("\n--- New Rule Data ---")
import json
print(json.dumps(result.data, indent=2))

## 5. Create a Business Rule

In [ ]:
req = AgentRequest(
    query="create a business rule threshold for LTV:CAC ratio",
    context={
        "rule_name": "LTV:CAC Ratio Minimum",
        "expression": "ltv_cac_ratio >= 3.0",
        "threshold": 3.0,
        "severity": "High",
        "asset": "ltv",
        "owner": "Data Science",
    },
    data_products=["ltv"],
)
result = agent.execute(req)
print(result.summary)

## 6. Evaluate Rules (without Databricks — shows ⚠️ message)

In [ ]:
req = AgentRequest(query="evaluate all rules")
result = agent.execute(req)
print(result.summary)
print("\nMetadata:", result.metadata)

## 7. Check Registry After Creates

In [ ]:
print(f"Registry now has {len(RULE_REGISTRY)} rules:")
for rid, r in RULE_REGISTRY.items():
    print(f"  {rid}: {r['name']} [{r['type']}]")

## 8. Rule Routing Keywords

In [ ]:
routing_tests = [
    "create rule for nulls",
    "add rule for completeness",
    "create a business rule threshold",
    "create a dq rule",
    "evaluate all rules",
    "check rules now",
    "list rules",
    "show rules",
    "all rules",
    "what rules exist",
]
for q in routing_tests:
    req = AgentRequest(query=q)
    result = agent.execute(req)
    print(f"  '{q[:50]}' → agent_name={result.agent_name}, success={result.success}")